# Part 5 (실습) — 첫 미니 프로젝트: 마케팅 카드 생성기

> **이 노트북의 목표**
> 초급(Part 1~4)의 네 부품을 합쳐 작동하는 체인을 처음부터 끝까지 만듦. 상품명 → 구조화된 마케팅 카드. 마지막엔 "이 체인으로 풀기 어려운 요구"를 직접 마주하며 중급으로 넘어감.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~4 (졸업 작품)

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
# ChatPromptTemplate: 채팅 모델용 메시지 템플릿을 만드는 클래스
#   → .from_messages([(역할, 템플릿), ...]): 역할별 메시지 리스트 생성
#   → 역할: "system"(시스템), "human"(사용자), "ai"(모델 응답)
from langchain_core.prompts import ChatPromptTemplate
# StrOutputParser: AIMessage 객체에서 .content(텍스트)만 꺼내주는 파서
#   → 체인의 마지막에 붙여서 순수 문자열 결과를 얻을 때 사용
#   → 사용법: prompt | llm | StrOutputParser()
from langchain_core.output_parsers import StrOutputParser
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.7)
print("준비 완료")

## 1. 프로젝트 설계 — 입력과 출력 먼저

코드 전에 입력·출력을 못 박음.

| 입력 | 출력 |
|---|---|
| 상품명 (str) | 헤드라인·설명·타겟·해시태그·가격대 (구조화 객체) |

```
상품명 → [프롬프트] → [구조화 모델] → MarketingCard 객체
```

## 2. 1단계 — 출력 구조 설계 (Pydantic)

가장 먼저 "카드의 칸"을 정의함. 이것이 설계도임.

In [ ]:
# pydantic: 데이터 검증 라이브러리 — 입출력 데이터의 형식을 엄격하게 정의하고 검증
#   → BaseModel: 데이터 구조(스키마)를 정의하는 기본 클래스
#   → Field: 각 필드의 설명·기본값 등 메타데이터를 추가할 때 사용
from pydantic import BaseModel, Field

class MarketingCard(BaseModel):
    headline: str = Field(description="시선을 끄는 한 줄 카피")
    description: str = Field(description="상품 설명 2~3문장")
    target: str = Field(description="핵심 타겟 고객층")
    hashtags: list[str] = Field(description="SNS 해시태그 3~5개")
    price_range: str = Field(description="추천 가격대 (예: 5만원대)")

print("설계도 완성:", list(MarketingCard.model_fields.keys()))

## 3. 2단계 — 체인 조립 (프로젝트의 심장)

Part 2·3·4가 한 줄에 모임: `프롬프트 | 모델.with_structured_output(카드)`

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 리리마켓의 베테랑 마케터야. 20~40대 여성 고객을 잘 안다."),
    ("human", "다음 상품의 마케팅 카드를 만들어줘: {product}"),
])

# .with_structured_output(): 모델 응답을 정해진 형식(Pydantic 모델)으로 강제 변환
#   → JSON이 아닌 Python 객체로 바로 받을 수 있어 후처리가 편리함
chain = prompt | model.with_structured_output(MarketingCard)   # ← 핵심 한 줄

# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
card = chain.invoke({"product": "베이지 케이블 니트"})
print("타입:", type(card).__name__)
print("헤드라인:", card.headline)
print("설명    :", card.description)
print("타겟    :", card.target)
print("해시태그:", " ".join(card.hashtags))
print("가격대  :", card.price_range)

### 카드를 보기 좋게 출력하는 함수
객체라 자유롭게 다룰 수 있음.

In [ ]:
def print_card(c: MarketingCard):
    print("┌" + "─"*42)
    print(f"│ 📣 {c.headline}")
    print("├" + "─"*42)
    print(f"│ {c.description}")
    print(f"│ 🎯 타겟: {c.target}")
    print(f"│ 💰 {c.price_range}")
    print(f"│ {' '.join('#'+h.lstrip('#') for h in c.hashtags)}")
    print("└" + "─"*42)

print_card(card)

## 4. 3단계 — 확장: 배치 처리

체인은 다시 Runnable이라 `.batch()`로 여러 상품을 한꺼번에. (Part 3)

In [ ]:
products = [
    {"product": "트렌치 코트"},
    {"product": "린넨 셔츠"},
    {"product": "캐시미어 머플러"},
]

# .batch(): 여러 입력을 한꺼번에 보내 동시에 처리 (공동구매 방식)
#   → 리스트로 여러 질문을 넘기면 결과도 리스트로 반환
cards = chain.batch(products)   # 카드 객체 리스트
for c in cards:
    print_card(c)
    print()

## 5. 3단계(이어서) — 확장: 병렬 변형

한 상품에 대해 감성/실용 두 톤의 헤드라인을 동시에. (RunnableParallel)

In [ ]:
# ─── LCEL(LangChain Expression Language) Runnable 부품 ───
# RunnableParallel: 여러 작업을 동시에(병렬로) 실행하는 부품
#   → 입력을 여러 체인에 동시에 보내고 결과를 딕셔너리로 합침
from langchain_core.runnables import RunnableParallel

감성 = ChatPromptTemplate.from_template("{product}의 감성적 헤드라인 한 줄만") | model | StrOutputParser()
실용 = ChatPromptTemplate.from_template("{product}의 실용적 장점 헤드라인 한 줄만") | model | StrOutputParser()

variants = RunnableParallel(감성=감성, 실용=실용)
res = variants.invoke({"product": "울 코트"})
print("감성 버전:", res["감성"].strip())
print("실용 버전:", res["실용"].strip())

### 저장까지 — 객체를 dict로
`model_dump()`로 딕셔너리화하면 DB·스프레드시트 저장이 쉬움.

In [ ]:
# json: 파이썬 내장 모듈 — JSON 형식 데이터를 딕셔너리로 변환(파싱)하거나 그 반대를 할 때 사용
import json
card_dict = card.model_dump()
print(json.dumps(card_dict, ensure_ascii=False, indent=2))

## 6. 졸업 — 그리고 체인의 한계 마주하기

우리 체인은 강력하지만 **항상 정해진 길로만** 흐름. 다음 요구를 보자.

> "신상이면 **경쟁사 가격을 웹에서 검색**해 가격대를 정하고, **재고가 없으면 카드 대신 입고 알림 문구**를 만들어줘."

이건 '상황을 보고 다음 행동을 그때그때 고르는' 일 — 검색할지 판단하고, 카드/알림을 분기해야 함. 정해진 길(체인)로는 담을 수 없음.

In [ ]:
# 시도해보면 — 체인은 "검색"이나 "분기"를 스스로 못 함.
# 모델에게 말로 부탁해도, 진짜 웹 검색을 하거나 실행 흐름을 바꾸진 못함.
tricky = model.invoke(
    "베이지 니트의 경쟁사 실시간 가격을 검색해서 알려줘. (단, 너는 검색 도구가 없다)"
)
print(tricky.content)
print("\n→ 모델은 '검색할 수 없다'고 답하거나 지어냄. 진짜 행동(도구 사용)이 필요함.")

### 결론
- 정해진 순서의 일 → **체인** (지금까지 배운 것, 여전히 빠르고 안전함)
- 상황 판단·도구 사용·분기가 필요한 일 → **에이전트** (중급의 시작)

체인은 버려지지 않음 — 중급의 에이전트도 내부에서 이런 체인을 도구처럼 씀.

## 정리 — 초급 졸업 🎓

| 단계 | 한 일 | 쓴 부품 |
|---|---|---|
| 설계 | 입력·출력 정의 | — |
| 1 | Pydantic 카드 정의 | Part 4 |
| 2 | `프롬프트 | 모델.with_structured_output` | Part 2·3·4 |
| 3 | `.batch()` · `RunnableParallel` 확장 | Part 3 |
| 4 | 체인의 한계 마주 | → 중급 예고 |

### 3줄 요약
1. 초급의 네 부품을 합쳐 작동하는 마케팅 카드 생성기를 완성함.
2. 새 개념 없이 배운 도구의 조합만으로 배치·병렬까지 확장됨.
3. 체인은 정해진 길만 감 — 판단·도구·분기가 필요하면 에이전트(중급)로.

### ▶ 다음 (Part 6 · 중급 시작)
"체인 vs 에이전트" — 오늘 마주한 경계를 정면으로 다룸. 무엇이 정해진 길이고 무엇이 동적 판단인지 손에 잡히게 익힘.